# Automated AI Bakery Verification

This notebook will use the procedure developed in the bakery-verification branch to more strictly apply a bakery definition and filter bakery candidates from our intial model using Gemini API. The bakery candidates will be processed in descending ranks and each verification will be saved and considered for the overall verification after some manual review. Businesses will be processed in API batches and the process will be resumed across multiple days due to API rate limits. Previous verification results are loaded automatically so that businesses with completed verification are not resubmitted. The results will later be included in a final bakery name classification results csv.

In [26]:
from pathlib import Path
from getpass import getpass
from datetime import datetime

import json
import time
import pandas as pd

from google import genai
from google.genai import types, errors

# Verification settings
MODEL = "gemini-2.5-flash"

BUSINESSES_PER_REQUEST = 10
MAX_REQUESTS_PER_RUN = 140
REQUEST_DELAY_SECONDS = 2

# Paths
FHRS_DATA_DATE = "2026-07-23"

DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = (DATA_FOLDER / "ai_verification_v3")
VERIFICATION_FOLDER.mkdir(parents=True, exist_ok=True)

RANKED_FHRS_PATH = (DATA_FOLDER / f"london_fhrs_ranked_establishments_{FHRS_DATA_DATE}.csv")
RAW_FHRS_PATH = Path(f"../data/business/raw/london_fhrs_raw_{FHRS_DATA_DATE}.csv")

RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
BATCH_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_batches.jsonl")
ERROR_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_errors.jsonl")

# Loading dataset and checking recall range

Since verification via Gemini API can be costly, I found that choosing a recall threshold based on the classifier findings would be a more reasonable way to verify the business names.

In [6]:
fhrs_ranked = pd.read_csv(RANKED_FHRS_PATH)

address_columns = ["FHRSID", "AddressLine1", "AddressLine2", "AddressLine3", "AddressLine4"]

fhrs_raw_address = pd.read_csv(RAW_FHRS_PATH, usecols=address_columns)

fhrs_raw_address = (fhrs_raw_address.drop_duplicates(subset="FHRSID"))

fhrs_ranked = fhrs_ranked.merge(fhrs_raw_address, on="FHRSID", how="left", validate="many_to_one")

fhrs_ranked["Address"] = fhrs_ranked[["AddressLine1", "AddressLine2", "AddressLine3", "AddressLine4"]
                                     ].apply(lambda row: ", ".join(str(value).strip()
                                             for value in row 
                                             if pd.notna(value) and str(value).strip()), 
                                             axis=1)

verification_candidates = (
    fhrs_ranked[fhrs_ranked["BakeryRank"].notna()]
    .sort_values(["BakeryRank", "FHRSID"])
    .drop_duplicates(subset="BusinessNameClean")
    .rename(columns={"FHRSID": "FHRSIDRep"})
    .reset_index(drop=True))

# Candidate-selection threshold from classifier cross-validation (notebook_07)
RECALL_THRESHOLDS = {
    0.90: 0.021,
    0.95: 0.015,
    0.99: 0.006}

threshold_summary = pd.DataFrame({
    "TargetRecall": RECALL_THRESHOLDS.keys(),
    "BakeryScoreThreshold": RECALL_THRESHOLDS.values()
})

threshold_summary["Candidates"] = [(verification_candidates["BakeryScore"] >= threshold).sum()
                                   for threshold in threshold_summary["BakeryScoreThreshold"]]

threshold_summary["CandidateIncreaseFrom90%"] = ((
    threshold_summary["Candidates"] / (threshold_summary.loc[threshold_summary["TargetRecall"] == 0.90,"Candidates"].iloc[0]) - 1
    )* 100).round(2)

threshold_summary["RecallGainFrom90%"] = (threshold_summary["TargetRecall"] - 0.90) * 100

display(threshold_summary)

,TargetRecall,BakeryScoreThreshold,Candidates,CandidateIncreaseFrom90%,RecallGainFrom90%
0,0.90,0.021,19902,0.00,0.0
1,0.95,0.015,28847,44.95,5.0
2,0.99,0.006,49147,146.95,9.0


From this we can see that from 90% recall to 95% recall we add ~45% more candidates for potenitally 5% improvement in recall. Meanwhile for 9% increase from 90% recall to 99% recall we would add ~147% of candidates, which seems unreasonable as we are more than doubling the candidates for a potential improvement in recall

In [7]:
TARGET_RECALL = 0.95
MIN_BAKERY_SCORE = RECALL_THRESHOLDS[TARGET_RECALL]

verification_queue = (verification_candidates[verification_candidates["BakeryScore"] >= MIN_BAKERY_SCORE]
    .sort_values("BakeryRank")
    .reset_index(drop=True)
)

print(f"""
Selected target recall: {TARGET_RECALL:.0%}
BakeryScore threshold: {MIN_BAKERY_SCORE}
Businesses to verify: {len(verification_queue)}""")


Selected target recall: 95%
BakeryScore threshold: 0.015
Businesses to verify: 28847


# Building the ai verification queue

Before running the AI-verification, the queue is reset to build the rows that make up the upcoming request. Details of the progress thus far are also mentioned, and it is built to ensure that the number of requests do not exceed the businesses remaining.

In [33]:
if RESULTS_PATH.exists():
    previous_results = pd.read_csv(RESULTS_PATH)

    duplicate_results = previous_results["BusinessNameClean"].duplicated().sum()
    if duplicate_results:
        print(f"Duplicate result rows found: {duplicate_results}")

    completed_names = set(previous_results["BusinessNameClean"].dropna())

else:
    previous_results = pd.DataFrame()
    completed_names = set()

remaining_queue = verification_queue[~verification_queue["BusinessNameClean"].isin(completed_names)].copy()
remaining_queue = (remaining_queue.sort_values("BakeryRank").reset_index(drop=True))

max_businesses_per_run = (BUSINESSES_PER_REQUEST * MAX_REQUESTS_PER_RUN)

run_queue = (remaining_queue
             .head(max_businesses_per_run)
             .copy()
             .reset_index(drop=True))

requests_planned = (len(run_queue) + BUSINESSES_PER_REQUEST- 1) // BUSINESSES_PER_REQUEST

print(
f"""Previously completed: {len(completed_names)}
Remaining businesses: {len(remaining_queue)}
Businesses selected this run: {len(run_queue)}
Maximum API requests this run: {requests_planned}""")

Duplicate result rows found: 10
Previously completed: 28847
Remaining businesses: 0
Businesses selected this run: 0
Maximum API requests this run: 0


# Gemini API Key

In [9]:
api_key = getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)

# Prompt building

The rows are validated briefly before being used in each prompt iteratively per establishment. The prompt uses the key rules including a role, adding context, giving examples and ensuring the output is clear.

In [10]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"
    return str(value)

def build_verification_prompt(batch):

    batch = batch.reset_index(drop=True)

    business_blocks = []

    for i, row in batch.iterrows():
        business_blocks.append(
            f"""
BUSINESS {i + 1}
Name: {clean_value(row["BusinessName"])}
Address: {clean_value(row["Address"])}
Postcode: {clean_value(row["PostCode"])}
Local authority: {clean_value(row["LocalAuthorityName"])}
FHRS type: {clean_value(row["BusinessType"])}
""".strip()
        )

    business_text = "\n\n".join(business_blocks)

    prompt = f"""
ROLE
    
You are classifying London food establishments for a study of bakery
provision.

Use Google Search to investigate the SPECIFIC supplied FHRS
establishment.

Assign exactly one class:

CORE_BAKERY
BAKERY_CAFE
GROCER_BAKERY
NON_BAKERY
UNCLEAR


==================================================
1. EVIDENCE STANDARD
==================================================

A positive bakery classification requires AFFIRMATIVE EVIDENCE.

Do not classify a business as CORE_BAKERY, BAKERY_CAFE or
GROCER_BAKERY merely because:

- its name contains "bakery", "patisserie", "bakes", etc.;
- FHRS gives it a particular business type;
- it sells or makes some baked food;
- another branch of the same brand is a bakery;
- bakery activity seems plausible.

The evidence must support the required characteristics of the
specific supplied establishment.

IMPORTANT:

"Absence of evidence is not evidence of absence" applies when deciding
between NON_BAKERY and UNCLEAR.

It does NOT justify a positive bakery classification.

Before assigning a positive class, verify the required positive
criteria below.


==================================================
2. IDENTIFY THE ESTABLISHMENT
==================================================

Use the name, address, postcode, local authority and current web
evidence together.

A matching address/postcode is strong evidence. Where location
information is incomplete, a distinctive business in the same local
area may still be a reasonable match if there is no conflicting
evidence.

Classify the supplied establishment, NOT the brand generally.

A company's production kitchen, warehouse or office does not become
a bakery retail location merely because the company operates bakery
shops elsewhere.

FHRS BusinessType is contextual information only and must not decide
the class by itself.


==================================================
3. HARD EXCLUSIONS
==================================================

If reliable evidence establishes any of the following, classify
NON_BAKERY regardless of the business name:

- permanently closed or inactive;
- home-based only;
- online or delivery-only;
- wholesale or production-only with no customer-facing retail;
- event/catering operation with no qualifying fixed retail site;
- travelling or changing-location market stall;
- roaming/mobile operation without a stable customer-facing location.

A fixed kiosk, permanent market unit, fixed counter, or permanently
stationed food truck/trailer CAN count as a stable retail location.

For a closed business, begin the reason with:

INACTIVE:


==================================================
4. CORE_BAKERY
==================================================

CORE_BAKERY is the strict principal bakery definition.

Assign CORE_BAKERY only when BOTH are supported:

A. The supplied establishment operates from a stable,
   customer-facing retail location.

B. Its principal identity or specialism is genuinely bakery-led.

Examples include:

- bread bakeries;
- artisan/sourdough bakeries;
- bakehouses;
- boulangeries;
- genuine patisseries;
- bakery chains;
- culturally specific bakeries;
- specialist bread businesses centred on naan, roti, simit or
  similar breads.

Do not require a British or European bakery style.

A specialist naan or roti business can qualify where bread production
and sale are essentially the business itself.

A restaurant/takeaway selling naan, roti, pide or bread alongside a
wider meal menu is NON_BAKERY.


SINGLE-PRODUCT RULE

A specialist snack/dessert concept is not CORE_BAKERY simply because
its product is baked or dough-based.

Normally NON_BAKERY:

- bespoke/celebration-cake-only businesses;
- cookie-only businesses;
- doughnut-only businesses;
- pretzel-only businesses;
- cinnamon-roll-only businesses;
- macaron-only businesses;
- churro-only businesses;
- waffle/crepe-only businesses;
- similar narrow snack or dessert concepts.

A specialist bread bakery or genuine patisserie with a recognisable
bakery/pastry range can still qualify.


==================================================
5. BAKERY_CAFE
==================================================

Assign BAKERY_CAFE only when BOTH are supported:

A. The establishment operates as a cafe, sandwich shop or similar
   customer-facing food-service business.

B. Current menu/product evidence shows that bakery goods form a
   SUBSTANTIAL and PERSISTENT part of what customers can buy.

This is deliberately broader than CORE_BAKERY.

A business does not need to be primarily a bakery.

Examples can include a cafe with a substantial ongoing range of
breads, pastries, cakes and related bakery goods.

One or two pastries, some cakes, bread with meals, pizza, naan, pide
or incidental desserts are NOT enough.

When uncertain, inspect a current menu, ordering page or product range
before assigning BAKERY_CAFE.


==================================================
6. GROCER_BAKERY
==================================================

Assign GROCER_BAKERY only when BOTH are supported:

A. The establishment is a supermarket, grocer, deli, convenience
   store or other general food retailer.

B. There is AFFIRMATIVE evidence of substantial bakery provision at
   that supplied location.

Qualifying evidence can include:

- a dedicated bakery section;
- an in-store bakery;
- an identifiable bakery concession;
- a broad persistent fresh bakery range;
- bakery products forming a substantial part of the retail offering.

Examples:

Supermarket with a substantial in-store bakery
-> GROCER_BAKERY

Independent grocer with a large fresh bread/pastry operation
-> GROCER_BAKERY

Deli with a substantial persistent bakery range
-> GROCER_BAKERY

Asda containing an operating Greggs concession
-> GROCER_BAKERY

The following evidence is NOT enough by itself:

- sells bread;
- sells pastries;
- sells baked goods;
- may contain baked goods;
- stocks packaged cakes or bread;
- a generic chain website says some branches have bakeries.

The bakery evidence must apply to the supplied location and be
substantial rather than incidental.


==================================================
7. NON_BAKERY AND UNCLEAR
==================================================

NON_BAKERY:

Use when the establishment has been reasonably identified and the
evidence shows that it does not satisfy any positive category.

Typical examples include:

- ordinary restaurants;
- pizzerias;
- pubs/hotels;
- restaurants making bread as part of a wider menu;
- cafes with only minor bakery offerings;
- retailers with only incidental bakery products;
- the single-product snack/dessert concepts listed above;
- any hard exclusion listed earlier.


UNCLEAR:

Use only where the relevant establishment, location or operating model
genuinely cannot be established from available evidence.

If a business appears bakery-like but the evidence does not establish
the required customer-facing operation, use UNCLEAR rather than
inventing positive evidence.

Do not use UNCLEAR merely because the distinction between two known
categories requires judgement.


==================================================
8. DECISION ORDER
==================================================

For each business:

1. Can the supplied establishment reasonably be identified?
   No -> UNCLEAR

2. Does a hard exclusion apply?
   Yes -> NON_BAKERY

3. Are BOTH CORE_BAKERY requirements affirmatively supported?
   Yes -> CORE_BAKERY

4. Are BOTH BAKERY_CAFE requirements affirmatively supported?
   Yes -> BAKERY_CAFE

5. Are BOTH GROCER_BAKERY requirements affirmatively supported?
   Yes -> GROCER_BAKERY

6. Otherwise:
   -> NON_BAKERY if evidence establishes the operating model
   -> UNCLEAR if the relevant operating model/location remains
      genuinely unresolved


==================================================
9. SOURCES
==================================================

Prefer current evidence such as:

- official business website;
- official menu/order page;
- official social media;
- current delivery platform;
- market/shopping-centre/operator website;
- credible current directories.

For positive classifications, actively look for evidence of BOTH the
business operation and the bakery offering.

Do not use the business name alone as evidence.


==================================================
10. OUTPUT
==================================================

Return exactly {len(batch)} lines.

Every line must contain exactly four pipe-separated fields:

business number | location match | classification | short reason

Location match:

YES
UNCLEAR

If location match is UNCLEAR, class must be UNCLEAR.

Examples:

1 | YES | CORE_BAKERY | Fixed specialist bread bakery with customer-facing retail at the supplied location.
2 | YES | BAKERY_CAFE | Cafe whose current menu shows a substantial persistent bakery range.
3 | YES | GROCER_BAKERY | Supermarket with a verified dedicated in-store bakery at this location.
4 | YES | NON_BAKERY | Restaurant sells flatbread but bakery products are incidental to its wider menu.
5 | UNCLEAR | UNCLEAR | Available evidence cannot establish the operation at the supplied location.
6 | YES | NON_BAKERY | INACTIVE: The supplied establishment has permanently closed.

Return every business number exactly once.

Return ONLY the result lines.


==================================================
BUSINESSES
==================================================

{business_text}
""".strip()

    return prompt

# Verification functions

This is the function that will actually send completed prompts to Gemini 2.5 Flash with search grounding available to use web evidence when identifying establishments and assessing bakery activity. At a temperature of zero it reduces too much variation between requests and classifications are intended to be based on external evidence rather than the model's knowledge.

Grounding metadata is also extracted from the response to include source URLS, search queries and the amount of grounding support used by the API.

In [11]:
def verify_current_batch(batch):

    prompt = build_verification_prompt(batch)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0,
            max_output_tokens=1500,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            )
        )
    )

    return prompt, response

In [12]:
def get_grounding_info(response):

    if not response.candidates:
        return False, [], [], 0

    grounding = response.candidates[0].grounding_metadata

    if grounding is None:
        return False, [], [], 0

    search_queries = list(grounding.web_search_queries or [])

    source_urls = []

    for chunk in grounding.grounding_chunks or []:
        if chunk.web:
            source_urls.append(chunk.web.uri)

    source_urls = list(dict.fromkeys(source_urls))

    support_count = len(grounding.grounding_supports or [])

    grounding_used = bool(search_queries)

    return (grounding_used, search_queries, source_urls, support_count)

# Parsing and saving responses

The parser chekcs the structure of the responses from Gemini and only accepts known classification values along with correct location-match logic. These results are then appended onto the main verified CSV so that progress can continue for separate API runs.

Separate JSONL logs are produced for additional evidence from API requests with batch logs storing raw responses and search-grounding information. Error logs are also recorded including missing grounding, and incomplete responses alongside the obvious API errors.

In [ ]:
def parse_response(response_text, batch, batch_id):

    batch = batch.reset_index(drop=True)

    text = response_text.replace("\\_", "_").strip()

    # Gemini sometimes forgets newlines between results
    for number in range(len(batch), 0, -1):
        text = text.replace(f"{number} |", f"\n{number} |")

    allowed_classes = {
        "CORE_BAKERY",
        "BAKERY_CAFE",
        "GROCER_BAKERY",
        "NON_BAKERY",
        "UNCLEAR"
    }

    results = []
    parsed_numbers = []

    for line in text.splitlines():
        parts = [part.strip()
                for part in line.split("|", 3)]

        if len(parts) != 4:
            continue

        number_text = (parts[0].replace("*", "").strip())

        if not number_text.isdigit():
            continue

        business_number = int(number_text)

        if not 1 <= business_number <= len(batch):
            continue

        location_match = parts[1].upper()
        classification = parts[2].upper().replace(" ", "_")
        reason = parts[3]

        if location_match not in {"YES", "UNCLEAR"}:
            continue

        if classification not in allowed_classes:
            continue

        if location_match == "UNCLEAR" and classification != "UNCLEAR":
            continue

        row = batch.iloc[business_number - 1]

        results.append({
            "BusinessNameClean": row["BusinessNameClean"],
            "BusinessName": row["BusinessName"],
            "FHRSIDRep": row["FHRSIDRep"],
            "BusinessType": row["BusinessType"],
            "PostCode": row["PostCode"],
            "LocalAuthorityName": row["LocalAuthorityName"],
            "Address": row["Address"],
            "BakeryRank": row["BakeryRank"],
            "BakeryScore": row["BakeryScore"],
            "StoreCount": row["StoreCount"],

            "LocationMatch": location_match,
            "AIClass": classification,
            "AIReason": reason,

            "BatchID": batch_id,
            "Model": MODEL,
            "VerificationDateTime": datetime.now().isoformat(timespec="seconds")
        })

        parsed_numbers.append(business_number)

    missing_numbers = [number
                       for number in range(1, len(batch) + 1)
                       if number not in parsed_numbers]

    results = pd.DataFrame(results)

    return results, missing_numbers


def save_results(results):

    if results.empty:
        return

    results = results.drop_duplicates(subset="BusinessNameClean", keep="first")

    if RESULTS_PATH.exists():
        existing_names = set(pd.read_csv(RESULTS_PATH,
                                         usecols=["BusinessNameClean"]
                                         )["BusinessNameClean"].dropna())

        results = results[~results["BusinessNameClean"].isin(existing_names)]

    if results.empty:
        return 0


    results.to_csv(RESULTS_PATH, mode="a", header=not RESULTS_PATH.exists(), index=False)

    return len(results)


def save_jsonl(path, record):

    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False, default=str)+ "\n")

In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
requests_attempted = 0
results_saved = 0

for start in range(0, len(run_queue), BUSINESSES_PER_REQUEST):

    batch = (run_queue.iloc[start:start + BUSINESSES_PER_REQUEST]
             .copy()
             .reset_index(drop=True))

    requests_attempted += 1
    batch_id = f"{RUN_ID}_{requests_attempted}"

    try:
        prompt, response = verify_current_batch(batch)

        finish_reason = None
        finish_message = None

        if response.candidates:
            finish_reason = str(response.candidates[0].finish_reason)
            finish_message = (response.candidates[0].finish_message)

        grounding_used, search_queries, source_urls, support_count = (get_grounding_info(response))

        # Verification without grounding is not accepted
        if not grounding_used:
            save_jsonl(ERROR_LOG_PATH, {
                "BatchID": batch_id,
                "ErrorType": "NO_SEARCH_GROUNDING",
                "Businesses": batch["BusinessName"].tolist()})

            print(f"Batch {requests_attempted}: no Google Search grounding - not saved")

            time.sleep(REQUEST_DELAY_SECONDS)
            continue

        results, missing_numbers = (parse_response(response.text, batch, batch_id))

        # Save successful grounded results
        saved_count = save_results(results)
        results_saved += saved_count

        # Save the complete API request evidence
        save_jsonl(BATCH_LOG_PATH, {
            "BatchID": batch_id,
            "Time": datetime.now().isoformat(timespec="seconds"),
            "BusinessCount": len(batch),
            "ParsedCount": len(results),
            "BusinessNames": batch["BusinessName"].tolist(),
            "RawResponse": response.text,
            "GroundingUsed": grounding_used,
            "SearchQueries": search_queries,
            "SourceURLs": source_urls,
            "GroundingSupportCount": support_count,
            "Model": MODEL,
            "FinishReason": finish_reason,
            "FinishMessage": finish_message})

        # Record anything Gemini failed to return properly
        if missing_numbers:
            missing_businesses = [batch.iloc[number - 1]["BusinessName"]
                                  for number in missing_numbers]

            save_jsonl(ERROR_LOG_PATH, {
                "BatchID": batch_id,
                "ErrorType": "INCOMPLETE_RESPONSE",
                "Businesses": missing_businesses})

        print(f"Batch {requests_attempted}: saved {len(results)}/{len(batch)}")

    except errors.APIError as error:

        save_jsonl(ERROR_LOG_PATH, {
            "BatchID": batch_id,
            "ErrorType": "API_ERROR",
            "ErrorCode": error.code,
            "ErrorMessage": error.message,
            "Businesses": batch["BusinessName"].tolist()})

        print(f"Batch {requests_attempted} failed: {error.code}")

        if error.code == 429:
            print("Rate limit reached. Stopping safely.")
            break

        if 400 <= error.code < 500:
            print("Non-retryable API error. Stopping safely.")
            break

    except Exception as error:

        save_jsonl(ERROR_LOG_PATH, {
            "BatchID": batch_id,
            "ErrorType": type(error).__name__,
            "ErrorMessage": str(error),
            "Businesses": batch["BusinessName"].tolist()})

        print(f"Batch {requests_attempted} failed: {error}")

    time.sleep(REQUEST_DELAY_SECONDS)

Batch 1: saved 10/10
Batch 2: saved 10/10
Batch 3: saved 10/10
Batch 4: saved 10/10
Batch 5: saved 10/10
Batch 6: saved 10/10
Batch 7: saved 10/10
Batch 8: saved 10/10
Batch 9: saved 10/10
Batch 10: saved 10/10
Batch 11: saved 10/10
Batch 12: saved 10/10
Batch 13: saved 10/10
Batch 14: saved 10/10
Batch 15: saved 10/10
Batch 16: saved 10/10
Batch 17: saved 10/10
Batch 18: saved 10/10
Batch 19: saved 10/10
Batch 20: saved 10/10
Batch 21: saved 10/10
Batch 22: saved 10/10
Batch 23: saved 10/10
Batch 24: saved 10/10
Batch 25: saved 10/10
Batch 26: saved 10/10
Batch 27: saved 10/10
Batch 28: saved 10/10
Batch 29: saved 10/10
Batch 30: saved 10/10
Batch 31: saved 10/10
Batch 32: saved 10/10
Batch 33: saved 10/10
Batch 34: saved 10/10
Batch 35: saved 10/10
Batch 36: saved 10/10
Batch 37: saved 10/10
Batch 38: saved 10/10
Batch 39: saved 10/10
Batch 40: saved 10/10
Batch 41: saved 10/10
Batch 42: saved 10/10
Batch 43: saved 10/10
Batch 44: saved 10/10
Batch 45: saved 10/10
Batch 46: saved 10/

In [32]:
if RESULTS_PATH.exists():
    all_results = pd.read_csv(RESULTS_PATH)

    unique_results = (all_results
                      .sort_values("VerificationDateTime")
                      .drop_duplicates(subset="BusinessNameClean",
                                       keep="last")
                      .reset_index(drop=True))

    duplicate_count = len(all_results) - len(unique_results)

    print(f"""Requests attempted this run: {requests_attempted}
Results saved this run: {results_saved}
Raw result rows: {len(all_results)}
Duplicate rows: {duplicate_count}
Total businesses completed: {len(all_results)}""")

    print(f"\nClassifications:\n {all_results['AIClass'].value_counts()}")

    print(f"\nLocation matches:\n{all_results['LocationMatch'].value_counts()}")

Requests attempted this run: 139
Results saved this run: 1388
Raw result rows: 28857
Duplicate rows: 10
Total businesses completed: 28857

Classifications:
 AIClass
NON_BAKERY       19889
UNCLEAR           5677
GROCER_BAKERY     1335
BAKERY_CAFE       1321
CORE_BAKERY        635
Name: count, dtype: int64

Location matches:
LocationMatch
YES        23317
UNCLEAR     5540
Name: count, dtype: int64
